# The Explanation Is Not the Reason
### Fairwashing: a 96.7%-faithful explanation of a decision process that doesn't exist
**OWASP Boston — August 12, 2026** · Mardiros Merdinian

---
Same fraud model as the side-channel demo. Same defect.

This time we don't attack the model. We attack **the artifact the human trusts
to decide whether to trust the model.**

*OWASP Agentic Top 10 — ASI09 Human-Agent Trust Exploitation, "Fake Explainability."*

In [1]:
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

SEED = 1337; rng = np.random.default_rng(SEED); N = 60_000

# ---- population: instrument + tenure are LEGITIMATE risk signals ----------
established = rng.binomial(1, 0.62, N)
tenure_days = np.where(established == 1, rng.gamma(9, 130, N),
                                          rng.gamma(1.6, 55, N)).clip(1, 4000)
instrument  = np.where(established == 1, rng.choice([1, 2], N, p=[.30, .70]),
                                          rng.choice([0, 1], N, p=[.68, .32]))
amount          = np.exp(rng.normal(3.9, 1.0, N)).clip(1, 5000)
hour            = rng.integers(0, 24, N)
bill_ship_match = rng.binomial(1, .86, N)
device_reuse    = rng.poisson(1.1, N)
velocity_1h     = rng.poisson(.55, N)
email_age_days  = np.where(established == 1, rng.gamma(3.0, 300, N),
                           rng.gamma(1.4, 110, N)).clip(0, 4000)

# ---- TRUE fraud: driven by behaviour, not by what card you hold -----------
z = (-3.35 + 1.35*(bill_ship_match == 0) + 0.95*(velocity_1h >= 2)
     + 0.80*(device_reuse >= 3) + 0.90*(np.log(amount) > 5.6)
     + 0.75*(email_age_days < 60) + 0.35*((hour >= 1) & (hour <= 5))
     - 0.22*established + rng.normal(0, 0.30, N))
true_fraud = rng.binomial(1, 1/(1+np.exp(-z)))

FULL = ["amount","hour","tenure_days","instrument","bill_ship_match",
        "device_reuse","velocity_1h","email_age_days"]
SAN  = ["amount","hour","bill_ship_match","device_reuse","velocity_1h","email_age_days"]

In [2]:
# THE DEFECT: labels come from INVESTIGATIONS, not from reality.
# The legacy rules engine looked at thin-file 74% of the time, established 16%.
p_investigated = np.where(established == 1, 0.16, 0.74)
observed_label = true_fraud * rng.binomial(1, p_investigated)

df = pd.DataFrame(dict(amount=amount, hour=hour, tenure_days=tenure_days,
    instrument=instrument, bill_ship_match=bill_ship_match, device_reuse=device_reuse,
    velocity_1h=velocity_1h, email_age_days=email_age_days,
    established=established, true_fraud=true_fraud, observed_label=observed_label))
sp = rng.random(N) < 0.7; train, test = df[sp].copy(), df[~sp].copy()

def fit(f, label, feats=FULL, w=None):
    m = HistGradientBoostingClassifier(max_iter=180, learning_rate=.08,
                                       max_depth=6, random_state=SEED)
    m.fit(f[feats], f[label], sample_weight=w); return m

model = fit(train, "observed_label")
test["score"] = model.predict_proba(test[FULL])[:, 1]
T0 = np.quantile(test.score, .93)          # ~7% decline budget
test["d0"] = (test.score >= T0).astype(int)

print(f"true fraud rate   | thin-file {true_fraud[established==0].mean()*100:5.2f}%"
      f"   established {true_fraud[established==1].mean()*100:5.2f}%")
print(f"LABELLED fraud    | thin-file {observed_label[established==0].sum():5d}"
      f"       established {observed_label[established==1].sum():5d}")

true fraud rate   | thin-file  7.97%   established  5.55%
LABELLED fraud    | thin-file  1332       established   320


### The reviewer's question
A thin-file customer is declined and complains. A human reviewer asks the
system: **why?** The explanation must not cite tenure or payment instrument —
those are the features compliance told us to keep out of customer-facing
rationales.

So we fit a surrogate on the *sanitized* features that reproduces the model's decisions.

In [3]:
from sklearn.tree import DecisionTreeClassifier, export_text

surrogate = DecisionTreeClassifier(max_depth=5, random_state=SEED).fit(test[SAN], test["d0"])
pred = surrogate.predict(test[SAN])

print(f"  fidelity to the real model's decisions : {(pred == test.d0).mean()*100:5.1f}%")
print(f"  declines correctly reproduced          : {pred[test.d0==1].mean()*100:5.1f}%")
print(f"  features used                          : {', '.join(SAN)}")
print(f"  features NOT used                      : tenure_days, instrument")

  fidelity to the real model's decisions :  96.7%
  declines correctly reproduced          :  66.3%
  features used                          : amount, hour, bill_ship_match, device_reuse, velocity_1h, email_age_days
  features NOT used                      : tenure_days, instrument


### What the reviewer sees

In [4]:
print(export_text(surrogate, feature_names=SAN, max_depth=2))

|--- bill_ship_match <= 0.50
|   |--- email_age_days <= 167.58
|   |   |--- amount <= 10.71
|   |   |   |--- truncated branch of depth 3
|   |   |--- amount >  10.71
|   |   |   |--- truncated branch of depth 3
|   |--- email_age_days >  167.58
|   |   |--- email_age_days <= 421.39
|   |   |   |--- truncated branch of depth 3
|   |   |--- email_age_days >  421.39
|   |   |   |--- truncated branch of depth 3
|--- bill_ship_match >  0.50
|   |--- email_age_days <= 53.63
|   |   |--- device_reuse <= 2.50
|   |   |   |--- truncated branch of depth 3
|   |   |--- device_reuse >  2.50
|   |   |   |--- truncated branch of depth 3
|   |--- email_age_days >  53.63
|   |   |--- device_reuse <= 2.50
|   |   |   |--- truncated branch of depth 3
|   |   |--- device_reuse >  2.50
|   |   |   |--- truncated branch of depth 3



Billing/shipping mismatch. Email account age. Amount. Velocity.

Every one of these is a defensible, sanctioned fraud signal. Nothing in this
explanation is a lie about the *outputs* — it matches them 96.7% of the time.

---
### The counterfactual
Take every declined thin-file transaction. Change **only** the two things the
explanation never mentions. Change nothing the explanation *does* mention.

In [5]:
dec = test[(test.d0 == 1) & (test.established == 0)].copy()
cf  = dec.copy(); cf["tenure_days"] = 900.0; cf["instrument"] = 2

flipped = model.predict_proba(cf[FULL])[:, 1] < T0
print(f"  declined thin-file transactions            : {len(dec):5d}")
print(f"  that flip to APPROVED on tenure/instrument : {flipped.mean()*100:5.1f}%")
print(f"\n  ...none of which appears anywhere in the explanation.")

  declined thin-file transactions            :  1212
  that flip to APPROVED on tenure/instrument :  98.0%

  ...none of which appears anywhere in the explanation.


---
### The explanation is 96.7% faithful and 98% wrong

It accurately predicts *what* the model decides. It describes a *reason* that
isn't the reason. The reviewer reads it, finds it plausible, and closes the case
— and the actual decisive factor was never on the page.

**This is the mechanism, one abstraction layer below an agent fabricating 4,000
users to explain a database it deleted.** The human's oversight didn't fail
because they were careless. It failed because the artifact they were handed to
oversee *with* was constructed by the thing being overseen.

*Opaque systems stifle independent human judgment by encouraging deference to
AI-generated conclusions.* — Vallor, arriving at ASI09 from virtue ethics.